## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import duckdb

from sklearn.decomposition import PCA
from sklearn.cluster import MiniBatchKMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.utils import shuffle

import umap.umap_ as umap

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib import colormaps


## 1. Load the Taxi Data


In [ ]:
from pathlib import Path

PARQUET_PATH = 'combined_2024_with_features_with_weather.parquet'
SAMPLE_SIZE  = 1_000_000
RANDOM_STATE = 42

FEATURE_COLS = [
    'trip_distance_mi',
    'trip_duration_min',
    'speed_mph',
    'fare',
    'price_per_mi',
    'fare_per_minute',
    'pickup_hour',
    'day_of_week',
    'is_weekend',
    'is_rush_hour',
    'is_public_holiday',
    'pickup_count_per_slot_in_zone',
    'dropoff_count_per_slot_in_zone',
    'net_zone_flow',
    'HourlyDryBulbTemperature',
    'HourlyDewPointTemperature',
    'HourlyWetBulbTemperature',
    'HourlyPrecipitation',
    'HourlyWindSpeed',
    'DailySnowDepth',
    'DailySnowfall',
    'DailyPrecipitation',
    'DailyPeakWindSpeed',
    'DailyAverageDryBulbTemperature',
    'DailyAverageDewPointTemperature',
    'DailyAverageWetBulbTemperature',
    'DailyAverageWindSpeed',
    'DailyMaximumDryBulbTemperature',
    'DailyMinimumDryBulbTemperature',
    'DailyHeatingDegreeDays',
    'DailyCoolingDegreeDays',
]

LABEL_COLS = [
    'cab_type',
    'time_of_day_bin',
    'is_rush_hour',
    'is_weekend',
    'HourlyPresentWeatherType',
    'DailyWeather',
]

ALL_COLS = list(dict.fromkeys(FEATURE_COLS + LABEL_COLS))
_parquet = str(Path(PARQUET_PATH).resolve())

print(f'Sampling from {_parquet} ...')
_sql = f"""
    SELECT {', '.join(f'"{c}"' for c in ALL_COLS)}
    FROM read_parquet(?)
    USING SAMPLE {int(SAMPLE_SIZE)} ROWS
"""
df = duckdb.execute(_sql, [_parquet]).fetchdf()
df = shuffle(df, random_state=RANDOM_STATE).reset_index(drop=True)
print(f'Working sample: {len(df):,} rows x {df.shape[1]} columns')

null_frac = df[FEATURE_COLS].isnull().mean().sort_values(ascending=False)
print('\nTop null fractions:', null_frac.head(8).round(3).to_dict())

hourly_to_daily_map = {
    'HourlyDryBulbTemperature':  'DailyAverageDryBulbTemperature',
    'HourlyDewPointTemperature': 'DailyAverageDewPointTemperature',
    'HourlyWetBulbTemperature':  'DailyAverageWetBulbTemperature',
    'HourlyWindSpeed':           'DailyAverageWindSpeed',
    'HourlyPrecipitation':       'DailyPrecipitation',
}
for hourly_col, daily_col in hourly_to_daily_map.items():
    df[hourly_col] = df[hourly_col].fillna(df[daily_col])
    df[daily_col]  = df[daily_col].fillna(df[hourly_col])

print('\nTop null fractions after cross-fill:',
      df[FEATURE_COLS].isnull().mean().sort_values(ascending=False).head(8).round(3).to_dict())

# Fill remaining nulls

zero_fill_cols = [
    'HourlyPrecipitation', 'DailyPrecipitation',
    'DailySnowDepth', 'DailySnowfall',
]
for col in zero_fill_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)

median_fill_cols = [
    'HourlyDryBulbTemperature', 'HourlyDewPointTemperature',
    'HourlyWetBulbTemperature', 'HourlyWindSpeed',
    'DailyAverageDryBulbTemperature', 'DailyAverageDewPointTemperature',
    'DailyAverageWetBulbTemperature', 'DailyAverageWindSpeed',
    'DailyMaximumDryBulbTemperature', 'DailyMinimumDryBulbTemperature',
    'DailyPeakWindSpeed', 'DailyHeatingDegreeDays', 'DailyCoolingDegreeDays',
]
for col in median_fill_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        df[col] = df[col].fillna(df[col].median())

rn = df[FEATURE_COLS].isnull().sum()
print('\nRemaining nulls in features:', int(rn.sum()))
if rn.sum():
    print(rn[rn > 0])

## 2. Plotting helpers

In [ ]:
def make_color_map(labels):
    """Map unique string/int labels to colours from the tab10 palette."""
    unique = sorted(set(labels))
    palette = colormaps['tab10'].colors
    return {val: palette[i % len(palette)] for i, val in enumerate(unique)}


def scatter2d(X, labels, title='', alpha=0.3, s=2, ax=None,
              xlabel='Component 1', ylabel='Component 2', legend_fontsize=10):
    """2D scatter coloured by labels; pass ax for subplot layouts."""
    color_map = make_color_map(labels)
    point_colors = [color_map[l] for l in labels]
    own_fig = ax is None
    if own_fig:
        _, ax = plt.subplots(figsize=(14, 9))
    ax.scatter(X[:, 0], X[:, 1], c=point_colors, alpha=alpha, s=s)
    patches = [mpatches.Patch(color=col, label=str(lbl))
                 for lbl, col in color_map.items()]
    ax.legend(handles=patches, loc='best', fontsize=legend_fontsize, markerscale=4)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if own_fig:
        plt.tight_layout()
        plt.show()


def scatter2d_numeric(X, values, title='', cmap='viridis', alpha=0.3, s=2):
    """
    2D scatter plot coloured by a continuous numeric value (e.g. speed, fare).
    Useful for seeing gradients across the embedding.
    """
    fig, ax = plt.subplots(figsize=(14, 9))
    sc = ax.scatter(X[:, 0], X[:, 1], c=values, cmap=cmap, alpha=alpha, s=s)
    plt.colorbar(sc, ax=ax)
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('Component 1')
    ax.set_ylabel('Component 2')
    plt.tight_layout()
    plt.show()

## 3. Feature matrix

In [ ]:
X_raw = df[FEATURE_COLS].values
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print(f'Feature matrix {X.shape}; scaled mean max |.| {np.abs(X.mean(axis=0)).max():.4f}; '
      f'std range {X.std(axis=0).min():.3f}–{X.std(axis=0).max():.3f}')

## 4. PCA

### 4a. Scree plot

In [ ]:
pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X)

evr        = pca_full.explained_variance_ratio_
cumulative = evr.cumsum()

print(f'Explained variance — first 5 PCs: {np.round(evr[:5], 4)}')
print(f'Last PC EVR: {evr[-1]:.6f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.bar(range(1, len(evr)+1), evr, color='blue')
ax1.set_xlabel('Principal Component')
ax1.set_ylabel('Explained Variance Ratio')
ax1.set_title('Scree Plot — Variance per Component')

ax2.plot(range(1, len(cumulative)+1), cumulative, marker='o', color='orange')
ax2.axhline(0.90, color='red', linestyle='--', label='90% threshold')
ax2.axhline(0.95, color='green', linestyle='--', label='95% threshold')
ax2.set_xlabel('Number of Components')
ax2.set_ylabel('Cumulative Variance')
ax2.set_title('Cumulative Variance')
ax2.legend()

plt.tight_layout()
plt.show()

n_90 = np.argmax(cumulative >= 0.90) + 1
n_95 = np.argmax(cumulative >= 0.95) + 1
print(f'Components needed for 90% variance: {n_90}')
print(f'Components needed for 95% variance: {n_95}')

### 4b. PCA 2D

In [ ]:
pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X)

print(f'Variance explained by 2 components: {pca_2d.explained_variance_ratio_.sum():.3f}')

scatter2d(X_pca_2d, df['cab_type'].values,
          title='PCA (2D) — by cab type')

scatter2d(X_pca_2d, df['time_of_day_bin'].values,
          title='PCA (2D) — by time of day')

scatter2d_numeric(X_pca_2d, df['speed_mph'].values,
                  title='PCA (2D) — by speed_mph', cmap='coolwarm')

scatter2d_numeric(X_pca_2d, df['fare'].values,
                  title='PCA (2D) — by fare', cmap='YlOrRd')

### 4c. PCA for UMAP input

In [ ]:
N_PCA_COMPONENTS = n_90 

pca_pre = PCA(n_components=N_PCA_COMPONENTS, random_state=RANDOM_STATE)
X_pca   = pca_pre.fit_transform(X)

print(f'PCA reduced shape: {X_pca.shape}')
print(f'Cumulative variance: {pca_pre.explained_variance_ratio_.sum():.4f}')

## 5. UMAP

In [ ]:
%%time
UMAP_SAMPLE = 200_000
idx = np.random.default_rng(RANDOM_STATE).choice(len(X_pca), size=UMAP_SAMPLE, replace=False)
X_pca_umap = X_pca[idx]
df_umap = df.iloc[idx].reset_index(drop=True)

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric='euclidean',
    low_memory=True,
    random_state=RANDOM_STATE,
    verbose=False,
)

X_umap = reducer.fit_transform(X_pca_umap)
print(f'UMAP embedding shape: {X_umap.shape}')

## 6. K-means clustering

### 6a. Choose k (elbow / silhouette)

In [ ]:
%%time
k_range    = range(2, 16)
inertias   = []
silhouettes = []

sil_sample_size = min(50_000, len(X_umap))
sil_idx = np.random.default_rng(RANDOM_STATE).choice(
    len(X_umap), size=sil_sample_size, replace=False
)

for k in k_range:
    model  = MiniBatchKMeans(n_clusters=k, random_state=RANDOM_STATE,
                             batch_size=10_000, n_init=3)
    labels = model.fit_predict(X_umap)
    inertias.append(model.inertia_)
    sil = silhouette_score(X_umap[sil_idx], labels[sil_idx])
    silhouettes.append(sil)
    print(f'  k={k:2d}  inertia={model.inertia_:>12.0f}  silhouette={sil:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(k_range, inertias, marker='o', color='blue')
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Inertia')
ax1.set_title('Elbow Method')

ax2.plot(k_range, silhouettes, marker='o', color='red')
ax2.set_xlabel('Number of clusters:')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Scores')

best_k = list(k_range)[np.argmax(silhouettes)]
ax2.axvline(best_k, color='red', linestyle='--', label=f'Best k={best_k}')
ax2.legend()

plt.tight_layout()
plt.show()
print(f'Silhouette shows best k = {best_k}')

### 6b. Fit K-means

In [ ]:
final_model = MiniBatchKMeans(
    n_clusters=best_k, random_state=RANDOM_STATE,
    batch_size=10_000, n_init=10,
)
cluster_labels = final_model.fit_predict(X_umap)

df_umap['cluster'] = cluster_labels
print(f'Cluster label distribution:')
print(df_umap['cluster'].value_counts().sort_index())

### 6c. Cluster plot

In [ ]:
scatter2d(X_umap, cluster_labels.astype(str),
          title=f'K-means clusters (k={best_k})',
          alpha=0.4, s=3)

### 6d. Cluster profiles

In [ ]:
profile_cols = [    
    'trip_distance_mi',
    'trip_duration_min',
    'speed_mph',
    'fare',
    'price_per_mi',
    'fare_per_minute',
    'pickup_hour',
    'day_of_week',
    'is_weekend',
    'is_rush_hour',
    'is_public_holiday',
    'pickup_count_per_slot_in_zone',
    'dropoff_count_per_slot_in_zone',
    'net_zone_flow',

    'HourlyDryBulbTemperature',
    'HourlyWetBulbTemperature',
    'HourlyPrecipitation',
    'HourlyWindSpeed',
    'DailySnowDepth',
    'DailySnowfall',
    'DailyPrecipitation',
    'DailyPeakWindSpeed',
    'DailyAverageDryBulbTemperature',
    'DailyAverageWetBulbTemperature',
    'DailyAverageWindSpeed', 
    
]

cluster_profile = df_umap.groupby('cluster')[profile_cols].mean().round(3)
print('Cluster profiles (mean of each feature per cluster):')
print(cluster_profile.T.to_string())

In [ ]:
profile_norm = (cluster_profile - cluster_profile.min()) / \
               (cluster_profile.max() - cluster_profile.min() + 1e-9)

fig, ax = plt.subplots(figsize=(14, 6))
im = ax.imshow(profile_norm.T.values, cmap='YlOrRd', aspect='auto')
plt.colorbar(im, ax=ax, label='Normalised mean (0=lowest, 1=highest)')
ax.set_xticks(range(best_k))
ax.set_xticklabels([f'Cluster {i}' for i in range(best_k)])
ax.set_yticks(range(len(profile_cols)))
ax.set_yticklabels(profile_cols, fontsize=9)
ax.set_title('Cluster Profile Heatmap — brighter = higher relative value')
plt.tight_layout()
plt.show()

### 6e. Clusters vs labels

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
scatter2d(X_umap, cluster_labels.astype(str),
          title=f'K-means clusters (k={best_k})', ax=axes[0],
          xlabel='UMAP 1', ylabel='UMAP 2', legend_fontsize=8)
scatter2d(X_umap, df_umap['cab_type'].values,
          title='Actual: cab_type', ax=axes[1],
          xlabel='UMAP 1', ylabel='UMAP 2', legend_fontsize=8)
scatter2d(X_umap, df_umap['time_of_day_bin'].values,
          title='Actual: time_of_day_bin', ax=axes[2],
          xlabel='UMAP 1', ylabel='UMAP 2', legend_fontsize=8)
plt.suptitle('K-means clusters vs known labels on UMAP embedding', fontsize=14)
plt.tight_layout()
plt.show()

## 7. Summary

In [ ]:
print('=== Cluster summary (means; % of full sample uses df row count) ===\n')
for c in range(best_k):
    row = cluster_profile.loc[c]
    size = (df_umap['cluster'] == c).sum()
    pct = 100 * size / len(df_umap)
    print(
        f'Cluster {c} — {size:,} trips ({pct:.1f}% of UMAP sample)\n'
        f'  speed {row["speed_mph"]:.1f} mph | fare ${row["fare"]:.2f} | '
        f'distance {row["trip_distance_mi"]:.2f} mi\n'
        f'  pickup hour {row["pickup_hour"]:.1f} | rush {100*row["is_rush_hour"]:.0f}% | '
        f'weekend {100*row["is_weekend"]:.0f}%\n'
        f'  zone pickups/slot {row["pickup_count_per_slot_in_zone"]:.1f} | '
        f'net zone flow {row["net_zone_flow"]:.1f}\n'
        f'  weather: dry-bulb {row["HourlyDryBulbTemperature"]:.1f}° | '
        f'precip {row["HourlyPrecipitation"]:.2f} | wind {row["HourlyWindSpeed"]:.1f}\n'
    )